In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, HistGradientBoostingRegressor

from sklearn.metrics import  r2_score, mean_squared_error, mean_absolute_error

import pickle

import warnings
warnings.filterwarnings("ignore")

# **Task 1: Data Loading**

In [2]:
city_attr = pd.read_csv("city_attributes.csv")
humidity = pd.read_csv("humidity.csv")
pressure = pd.read_csv("pressure.csv")
temperature = pd.read_csv("temperature.csv")
weather_desc = pd.read_csv("weather_description.csv")
wind_dir = pd.read_csv("wind_direction.csv")
wind_speed = pd.read_csv("wind_speed.csv")

print("Temperature CSV:")
print(temperature.head())
print(temperature.shape)

Temperature CSV:
              datetime   Vancouver    Portland  San Francisco     Seattle  \
0  2012-10-01 12:00:00         NaN         NaN            NaN         NaN   
1  2012-10-01 13:00:00  284.630000  282.080000     289.480000  281.800000   
2  2012-10-01 14:00:00  284.629041  282.083252     289.474993  281.797217   
3  2012-10-01 15:00:00  284.626998  282.091866     289.460618  281.789833   
4  2012-10-01 16:00:00  284.624955  282.100481     289.446243  281.782449   

   Los Angeles   San Diego   Las Vegas     Phoenix  Albuquerque  ...  \
0          NaN         NaN         NaN         NaN          NaN  ...   
1   291.870000  291.530000  293.410000  296.600000   285.120000  ...   
2   291.868186  291.533501  293.403141  296.608509   285.154558  ...   
3   291.862844  291.543355  293.392177  296.631487   285.233952  ...   
4   291.857503  291.553209  293.381213  296.654466   285.313345  ...   

   Philadelphia    New York    Montreal      Boston   Beersheba  \
0           NaN     

# **Task 2: Data Preprocessing**

1.   Melt wide to long format

In [3]:
def melt_df(df, val_name):
  return df.melt(id_vars=['datetime'], var_name='city', value_name=val_name)

temp = melt_df(temperature, "temperature")
humi = melt_df(humidity, "humidity")
pres = melt_df(pressure, "pressure")
wthr_desc = melt_df(weather_desc, "weather_description")
wind_d = melt_df(wind_dir, "wind_direction")
wind_s = melt_df(wind_speed, "wind_speed")

2. Merge all into one dataframe

In [4]:
df = temp.merge(humi, on=['datetime', 'city']) \
         .merge(pres, on=['datetime', 'city']) \
         .merge(wthr_desc, on=['datetime', 'city']) \
         .merge(wind_d, on=['datetime', 'city']) \
         .merge(wind_s, on=['datetime', 'city'])

df

,datetime,city,temperature,humidity,pressure,weather_description,wind_direction,wind_speed
0,2012-10-01 12:00:00,Vancouver,NaN,NaN,NaN,NaN,NaN,NaN
1,2012-10-01 13:00:00,Vancouver,284.630000,76.0,NaN,mist,0.0,0.0
2,2012-10-01 14:00:00,Vancouver,284.629041,76.0,NaN,broken clouds,6.0,0.0
3,2012-10-01 15:00:00,Vancouver,284.626998,76.0,NaN,broken clouds,20.0,0.0
4,2012-10-01 16:00:00,Vancouver,284.624955,77.0,NaN,broken clouds,34.0,0.0
...,...,...,...,...,...,...,...,...
1629103,2017-11-29 20:00:00,Jerusalem,NaN,NaN,NaN,NaN,NaN,NaN
1629104,2017-11-29 21:00:00,Jerusalem,NaN,NaN,NaN,NaN,NaN,NaN
1629105,2017-11-29 22:00:00,Jerusalem,NaN,NaN,NaN,NaN,NaN,NaN
1629106,2017-11-29 23:00:00,Jerusalem,NaN,NaN,NaN,NaN,NaN,NaN


3. Convert temperature from Kelvin to Celsius and remove outliers

In [5]:
df['temperature'] = df['temperature'] - 273.15
df = df[df['temperature'].between(-50, 60)]

df.shape

(1621078, 8)

4. Feature engineering from datetime

In [6]:
df['datetime'] = pd.to_datetime(df['datetime'])
df['year'] = df['datetime'].dt.year
df['month'] = df['datetime'].dt.month
df['day'] = df['datetime'].dt.day
df['hour'] = df['datetime'].dt.hour

5. Split features and target

In [7]:
X = df.drop(columns=['temperature','datetime','city'], axis=1)
y = df['temperature']

6. Preprocessing pipelines

In [8]:
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns
numeric_features

Index(['humidity', 'pressure', 'wind_direction', 'wind_speed'], dtype='object')

In [9]:
categorical_features = X.select_dtypes(include=['object']).columns
categorical_features

Index(['weather_description'], dtype='object')

In [10]:
num_transformer = Pipeline(
    steps = [
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ]
)

In [11]:
cat_transformer = Pipeline(
    steps = [
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ]
)

In [12]:
preprocessor = ColumnTransformer(
    transformers= [
        ('num', num_transformer, numeric_features),
        ('cat', cat_transformer, categorical_features)
    ]
)

# **TASK 3: Pipeline Creation**

In [13]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [14]:
X_train_small = X_train.sample(300_000, random_state=42)
y_train_small = y_train.loc[X_train_small.index]

X_test_small = X_test.sample(300_000, random_state=42)
y_test_small = y_test.loc[X_test_small.index]

In [15]:
reg_lr = LinearRegression()

reg_rf = RandomForestRegressor(
    n_estimators=50,
    n_jobs=-1,
    random_state=42
)

reg_gb = HistGradientBoostingRegressor(
    max_iter=100,
    learning_rate=0.1,
    max_depth=6,
    random_state=42
)

# **TASK 4: Primary Model Selection**

Selected Model: RandomForestRegressor

Justification: I chose Random Forest as the primary model because it can naturally capture non-linear patterns, handles noisy data well, and works effectively with high-dimensional tabular features. Additionally, I included Linear Regression and Gradient Boosting to compare performance and ensure the best possible prediction results.

# **TASK 5: Model Training**

In [16]:
model_to_train = {
    'Linear Regression' : reg_lr,
    'Random Forest' : reg_rf,
    'Gradient Boosting' : reg_gb
}

In [17]:
result = []

for name, model in model_to_train.items():
  pipe = Pipeline(
      [
          ('preprocessor', preprocessor),
          ('model', model)
      ]
  )

  if name == 'Linear Regression':
    X_tr, y_tr = X_train, y_train
    X_te, y_te = X_test, y_test
  else:
    X_tr, y_tr = X_train_small, y_train_small
    X_te, y_te = X_test_small, y_test_small

  # train
  pipe.fit(X_tr, y_tr)

  #predict
  y_pred = pipe.predict(X_te)

  # evaluate
  r2 = r2_score(y_te, y_pred)
  rmse = np.sqrt(mean_squared_error(y_te, y_pred))
  mae = mean_absolute_error(y_te, y_pred)

  result.append({
      "Model": name,
      "R2 Score" :r2,
      "RMSE": rmse,
      "MAE" : mae
  })

results_df = pd.DataFrame(result).sort_values("R2 Score", ascending=False)

print(results_df)

               Model  R2 Score      RMSE       MAE
1      Random Forest  0.413976  7.921174  5.845205
2  Gradient Boosting  0.389685  8.083679  6.314605
0  Linear Regression  0.161282  9.467587  7.593589


# **TASK 6: Cross-Validation**

In [18]:
best_model_name = results_df.iloc[0]['Model']
best_model_obj = model_to_train[best_model_name]

final_pipe = Pipeline(
    steps=[
        ('processor', preprocessor),
        ('model', best_model_obj)
    ]
)

cv_scores = cross_val_score(
    final_pipe,
    X_train_small,
    y_train_small,
    cv=5,
    scoring='neg_root_mean_squared_error'
)

print("Average RMSE:", -cv_scores.mean())
print("Standard Deviation:", cv_scores.std())

Average RMSE: 7.978356562752545
Standard Deviation: 0.038294618936451336


# **TASK 7: Hyperparameter Tuning**

In [19]:
param_grid = {
    'model__n_estimators': [50, 100],
    'model__max_depth': [10, 20, None],
    'model__min_samples_split': [2, 5]
}

grid = GridSearchCV(
    final_pipe,
    param_grid,
    cv=3,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    verbose = 2
)
grid.fit(X_train_small, y_train_small)

print("Best Params:", grid.best_params_)
print("Best RMSE:", -grid.best_score_)

Fitting 3 folds for each of 12 candidates, totalling 36 fits
Best Params: {'model__max_depth': 20, 'model__min_samples_split': 5, 'model__n_estimators': 100}
Best RMSE: 7.827931127309182


# **TASK 8: Best Model Selection**

In [20]:
best_model = grid.best_estimator_

best_model.fit(X_train, y_train)

Pipeline(steps=[('processor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  Index(['humidity', 'pressure', 'wind_direction', 'wind_speed'], dtype='object')),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  Index(['weather_description'], dtype='object'))])),
                ('model',
                 RandomForestRegressor(max_depth=20, min_samples_split=5,
                                       n_jobs=-1, random_state=42))])

# **TASK 9: Model Performance Evaluation**

In [21]:
y_final_pred = best_model.predict(X_test)

print("MAE:", mean_absolute_error(y_test, y_final_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_final_pred)))
print("R² Score:", r2_score(y_test, y_final_pred))

MAE: 5.683531011116252
RMSE: 7.568986837947875
R² Score: 0.4639407863195929


In [22]:
filename = "weather_temp_model.pkl"

with open(filename, "wb") as file:
  pickle.dump(best_model, file)